# World Cup Data — Visual Overview

Explore what StatsBomb event data looks like before any modeling: event types, spatial patterns, shot detail, and player stats.

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from mplsoccer import Pitch, VerticalPitch
from statsbombpy import sb

from src.data_loader import list_world_cup_seasons, get_matches, get_events

plt.rcParams.update({'figure.dpi': 120})

## 1. Available World Cup Seasons

In [ ]:
seasons = list_world_cup_seasons()
seasons[['season_id', 'season_name', 'competition_gender']].sort_values('season_name')

## 2. Pick a Season and Match

In [ ]:
target_season = '2022'  # change to any season_name shown above
season_row = seasons[seasons['season_name'] == target_season].iloc[0]
season_id = int(season_row['season_id'])
season_name = season_row['season_name']
print(f'Season: {season_name} World Cup  (season_id={season_id})')

matches = get_matches(season_id)
matches[['match_id', 'match_date', 'home_team', 'away_team', 'home_score', 'away_score']].sort_values('match_date')

In [ ]:
match_row = matches.iloc[0]  # change index to pick a different match
match_id = int(match_row['match_id'])
home = str(match_row['home_team'])
away = str(match_row['away_team'])
date = str(match_row['match_date'])
print(f'Match: {home} vs {away}  |  {date}')

events = get_events(match_id)
print(f'{len(events)} events, {events["type"].nunique()} distinct event types')
events.head(3)

## 3. Event Type Breakdown

Shows every event type StatsBomb tracks and how often each occurs in one match.

In [ ]:
type_counts = events['type'].value_counts()

fig, ax = plt.subplots(figsize=(9, 5))
type_counts.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Count')
ax.set_title(f'Event Types  —  {home} vs {away}', fontsize=13, pad=12)
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Shot Map

Bubble size = xG (how likely the model thinks the shot was to score). Stars = goals.

In [ ]:
shots = events[events['type'] == 'Shot'].copy()
shots['x'] = shots['location'].apply(lambda l: l[0])
shots['y'] = shots['location'].apply(lambda l: l[1])
shots['is_goal'] = shots['shot_outcome'] == 'Goal'

teams = shots['team'].unique()
team_colors = {teams[0]: '#3b82f6', teams[1]: '#ef4444'}

pitch = VerticalPitch(
    pitch_type='statsbomb', half=True,
    pitch_color='#0d1117', line_color='#8b949e', linewidth=1
)
fig, ax = pitch.draw(figsize=(7, 7))
fig.set_facecolor('#0d1117')

for team, grp in shots.groupby('team'):
    color = team_colors[team]
    non_goals = grp[~grp['is_goal']]
    pitch.scatter(
        non_goals['x'], non_goals['y'],
        s=non_goals['shot_statsbomb_xg'] * 1200 + 40,
        c=color, edgecolors='white', linewidths=0.8,
        alpha=0.6, ax=ax, zorder=3
    )
    goals = grp[grp['is_goal']]
    if len(goals):
        pitch.scatter(
            goals['x'], goals['y'],
            s=goals['shot_statsbomb_xg'] * 1200 + 60,
            c=color, edgecolors='white', linewidths=1.5,
            alpha=1.0, marker='*', ax=ax, zorder=4
        )

legend_handles = [
    mpatches.Patch(color=team_colors[teams[0]], label=teams[0]),
    mpatches.Patch(color=team_colors[teams[1]], label=teams[1]),
    Line2D([0], [0], marker='*', color='w', label='Goal', markersize=12, linestyle='None'),
    Line2D([0], [0], marker='o', color='w', label='Shot (size ∝ xG)', markersize=8, linestyle='None'),
]
ax.legend(handles=legend_handles, loc='lower center', ncol=2,
          facecolor='#1c2128', labelcolor='white', fontsize=9)
ax.set_title(f'Shot Map  ·  {home} vs {away}', color='white', fontsize=13, pad=10)
plt.show()

## 5. Touch Density by Team

KDE heatmap of all located events — shows territorial dominance and team shape.

In [ ]:
located = events.dropna(subset=['location']).copy()
located['x'] = located['location'].apply(lambda l: l[0])
located['y'] = located['location'].apply(lambda l: l[1])

pitch = Pitch(pitch_type='statsbomb', pitch_color='#0d1117', line_color='#8b949e', linewidth=1)
fig, axes = pitch.draw(nrows=1, ncols=2, figsize=(14, 5))
fig.set_facecolor('#0d1117')

axes_flat = list(axes.flat) if hasattr(axes, 'flat') else [axes]
for ax, (team, grp) in zip(axes_flat, located.groupby('team')):
    pitch.kdeplot(grp['x'], grp['y'], ax=ax, fill=True, cmap='plasma',
                  levels=80, alpha=0.85, thresh=0.05)
    ax.set_title(team, fontsize=12, color='white', pad=8)

fig.suptitle(f'Touch Density by Team  —  {home} vs {away}', fontsize=13, color='white', y=1.02)
plt.tight_layout()
plt.show()

## 6. What's Inside a Shot Event?

StatsBomb shot events are rich — outcome, body part, technique, type, and xG are all available.

In [ ]:
for label, col in [
    ('Outcome', 'shot_outcome'),
    ('Body part', 'shot_body_part'),
    ('Type (open play / free kick / etc.)', 'shot_type'),
    ('Technique', 'shot_technique'),
]:
    print(f'=== {label} ===')
    print(shots[col].value_counts().to_string())
    print()

In [ ]:
shot_detail = shots[[
    'player', 'team', 'minute', 'shot_outcome',
    'shot_statsbomb_xg', 'shot_technique', 'shot_body_part'
]].copy()
shot_detail.columns = ['Player', 'Team', 'Minute', 'Outcome', 'xG', 'Technique', 'Body Part']
shot_detail.sort_values('xG', ascending=False).reset_index(drop=True)

## 7. Season-Level Player Stats

Aggregate across a sample of matches. Increase `N_MATCHES` to cover more of the tournament (one API call per match).

In [ ]:
from src.player_stats import player_season_table

N_MATCHES = 15  # increase toward len(matches) for the full tournament
sample_matches = matches.head(N_MATCHES)

print(f'Fetching lineups for {N_MATCHES} matches...')
lineups_by_match = {
    int(row['match_id']): sb.lineups(match_id=int(row['match_id']))
    for _, row in sample_matches.iterrows()
}

season_stats = player_season_table(lineups_by_match)
print(f'{len(season_stats)} players found')
season_stats.sort_values('goals', ascending=False).head(10)

In [ ]:
qualified = season_stats[season_stats['minutes_played'] >= 45].copy()
top_goals = qualified.nlargest(12, 'goals')
top_xg = qualified.nlargest(12, 'xg')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].barh(top_goals['player_name'], top_goals['goals'], color='#3b82f6')
axes[0].invert_yaxis()
axes[0].set_title('Top Goalscorers', fontsize=12)
axes[0].set_xlabel('Goals')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

axes[1].barh(top_xg['player_name'], top_xg['xg'], color='#ef4444')
axes[1].invert_yaxis()
axes[1].set_title('Top xG', fontsize=12)
axes[1].set_xlabel('Expected Goals (xG)')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle(f'{season_name} World Cup — first {N_MATCHES} matches', fontsize=13)
plt.tight_layout()
plt.show()

## 8. xG vs Actual Goals

Points above the diagonal are overperforming their xG; points below are underperforming.

In [ ]:
scored = qualified[qualified['xg'] > 0].copy()

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(scored['xg'], scored['goals'], alpha=0.55, s=50,
           color='steelblue', edgecolors='#1e3a5f', linewidths=0.8)

max_val = max(scored['xg'].max(), scored['goals'].max()) * 1.05
ax.plot([0, max_val], [0, max_val], 'r--', lw=1.5, label='xG = Goals (perfect)')

outliers = scored[abs(scored['goals'] - scored['xg']) > 1.2]
for _, r in outliers.iterrows():
    name = r['player_name'].split()[-1]
    ax.annotate(name, (r['xg'], r['goals']), fontsize=8,
                xytext=(4, 4), textcoords='offset points', alpha=0.85)

ax.set_xlabel('Expected Goals (xG)', fontsize=11)
ax.set_ylabel('Actual Goals', fontsize=11)
ax.set_title('xG vs Actual Goals — over/underperformers', fontsize=12)
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()